In [1]:
import pandas as pd

df = pd.read_csv('nba_shot_data_2024_25_cleaned.csv')

In [2]:
import numpy as np

df['SHOT_ANGLE'] = np.degrees(np.arctan2(df['LOC_Y'], df['LOC_X']))

In [3]:
df['FINAL_5S_OF_PERIOD'] = (df['TIME_REMAINING_SEC'] <= 5).astype(int)
df['LATE_GAME'] = ((df['PERIOD'] >= 4) & (df['TIME_REMAINING_SEC'] <= 300)).astype(int)

In [4]:
def simplify_action(action):
    action = action.lower()
    if 'dunk' in action:
        return 'Dunk'
    elif 'layup' in action:
        return 'Layup'
    elif 'hook' in action:
        return 'Hook Shot'
    elif 'fadeaway' in action:
        return 'Fadeaway'
    else:
        return 'Jump Shot'

df['ACTION_TYPE_GROUPED'] = df['ACTION_TYPE'].apply(simplify_action)

In [5]:
categorical_columns = ['ACTION_TYPE_GROUPED', 'SHOT_ZONE_BASIC', 'SHOT_TYPE', 'SHOT_ZONE_AREA']
df_encoded = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

In [6]:
feature_columns = [
    'SHOT_DISTANCE', 'SHOT_ANGLE', 'PERIOD', 'TIME_REMAINING_SEC',
    'FINAL_5S_OF_PERIOD', 'LATE_GAME'
] + [col for col in df_encoded.columns if col.startswith((
    'SHOT_ZONE_BASIC_', 'SHOT_ZONE_AREA_', 'SHOT_TYPE_', 'ACTION_TYPE_GROUPED_' 
))]

In [7]:
x = df_encoded[feature_columns]
y = df_encoded['SHOT_MADE_FLAG']

In [8]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr, te = next(gss.split(x, y, groups=df_encoded['GAME_ID']))
x_train, x_test = x.iloc[tr], x.iloc[te]
y_train, y_test = y.iloc[tr], y.iloc[te]

assert set(df_encoded.iloc[tr]['GAME_ID']).isdisjoint(df_encoded.iloc[te]['GAME_ID'])

x_train.to_csv('x_train.csv', index=False)
x_test.to_csv('x_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)
print('X-test shape:', x_test.shape)
print('X-train shape:', x_train.shape)
print('Y-test shape:', y_test.shape)
print('Y-train shape:', y_train.shape)

X-test shape: (20230, 22)
X-train shape: (80962, 22)
Y-test shape: (20230,)
Y-train shape: (80962,)


In [9]:
import numpy as np

meta = df.reset_index(drop=True)[['GAME_ID', 'PLAYER_ID', 'PLAYER_NAME',
                                   'LOC_X', 'LOC_Y', 'SHOT_ZONE_BASIC']].copy()
meta['made'] = y.to_numpy()
meta['is_test'] = np.isin(np.arange(len(meta)), te)
assert len(meta) == len(x)
assert int(meta['is_test'].sum()) == len(x_test)
meta.to_csv('shots_meta.csv', index=False)
x.to_csv('x_all.csv', index=False)